In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTENC
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report

In [3]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

In [4]:
df_train['transaction_time'] = pd.to_datetime(df_train['transaction_time'])

df_train['year'] = df_train['transaction_time'].dt.year
df_train['month'] = df_train['transaction_time'].dt.month
df_train['day'] = df_train['transaction_time'].dt.day
df_train['hour'] = df_train['transaction_time'].dt.hour
df_train['minute'] = df_train['transaction_time'].dt.minute
df_train['day_of_week'] = df_train['transaction_time'].dt.dayofweek
df_test['transaction_time'] = pd.to_datetime(df_test['transaction_time'])

df_test['year'] = df_test['transaction_time'].dt.year
df_test['month'] = df_test['transaction_time'].dt.month
df_test['day'] = df_test['transaction_time'].dt.day
df_test['hour'] = df_test['transaction_time'].dt.hour
df_test['minute'] = df_test['transaction_time'].dt.minute
df_test['day_of_week'] = df_test['transaction_time'].dt.dayofweek

In [5]:
numeric_features = [
    'amount', 'ts_transaction_time', 'lat', 'lon', 'population_city', 'merchant_lat', 'merchant_lon'
]

cat_columns = [
    'merch', 'cat_id', 'name_1', 'name_2', 'gender', 'street', 'one_city', 'us_state', 'jobs', 'year',	'month',	'day',	'hour',	'minute'	,'day_of_week'
]

In [6]:
import numpy as np
from math import atan2, cos, radians, sin, sqrt


def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float, n_digits: int = 0) -> float:
    """
        Функция для расчёта расстояния от точки А до Б по прямой

        :param lat1: Широта точки А
        :param lon1: Долгота точки А
        :param lat2: Широта точки Б
        :param lon2: Долгота точки Б
        :param n_digits: Округляем полученный ответ до n знака после запятой
        :return: Дистанция по прямой с точностью до n_digits
    """

    lat1, lon1, lat2, lon2 = round(lat1, 6), round(lon1, 6), round(lat2, 6), round(lon2, 6)
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)

    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2

    return round(2 * 6372800 * np.arctan2(np.sqrt(a), np.sqrt(1 - a)), n_digits)  # метры.сантиметры


def bearing_degree(lat1: float, lon1: float, lat2: float, lon2: float, n_digits: int = 0) -> float:
    """
        Функция для расчёта угла между прямой [((lat1, lon1), (lat2, lon2)), (нулевой мередиан)]

        :param lat1: Широта точки А
        :param lon1: Долгота точки А
        :param lat2: Широта точки Б
        :param lon2: Долгота точки Б
        :param n_digits: Округляем полученный ответ до n знака после запятой
        :return: Значение угла с точностью до n_digits
    """

    lat1, lon1 = np.radians(round(lat1, 6)), np.radians(round(lon1, 6))
    lat2, lon2 = np.radians(round(lat2, 6)), np.radians(round(lon2, 6))

    dlon = (lon2 - lon1)
    numerator = np.sin(dlon) * np.cos(lat2)
    denominator = np.cos(lat1) * np.sin(lat2) - (np.sin(lat1) * np.cos(lat2) * np.cos(dlon))

    theta = np.arctan2(numerator, denominator)
    theta_deg = (np.degrees(theta) + 360) % 360

    return round(theta_deg, n_digits)

In [7]:
df_train['bearing_degree_1'] = bearing_degree(df_train['lat'], df_train['lon'], df_train['merchant_lat'], df_train['merchant_lon'], ).values
df_test['bearing_degree_1'] = bearing_degree(df_test['lat'], df_test['lon'], df_test['merchant_lat'], df_test['merchant_lon'], ).values
df_train['hav_dist_1'] = haversine_distance(df_train['lat'], df_train['lon'], df_train['merchant_lat'], df_train['merchant_lon'], ).values
df_test['hav_dist_1'] = haversine_distance(df_test['lat'], df_test['lon'], df_test['merchant_lat'], df_test['merchant_lon'], ).values

In [8]:
model_features = [
  'amount',
  'bearing_degree_1',
  #'bearing_degree_2',
  #'bearing_degree_3',
  'cat_id',
  'gender',
  'hav_dist_1',
  #'hav_dist_2',
  #'hav_dist_3',
  'jobs',
  'lat',
  'lon',
  'merch',
  #'merchant_lat',
  #'merchant_lon',
  'name_1',
  'name_2',
  'one_city',
  'population_city',
  #'post_code',
  'street',
  'us_state',
  'year',	
  'month',	
  'day',	
  'hour',	
  'minute',
  'day_of_week'
]

In [9]:
X = df_train.drop("target", axis=1) 
y = df_train["target"]

In [10]:
numeric_feat = ['amount','lat', 'lon', 'population_city', 'merchant_lat', 'merchant_lon', 'year',
       'month', 'day', 'hour', 'minute', 'day_of_week', 'bearing_degree_1',
       'hav_dist_1']

In [11]:
X = df_train.drop("merchant_lat", axis=1) 

In [12]:
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X[model_features], y)
print("Распределение классов после RandomOverSampler:")
print(pd.Series(y_train_res).value_counts())

Распределение классов после RandomOverSampler:
target
0    781927
1    781927
Name: count, dtype: int64


In [13]:

X_train, X_test, y_train, y_test = train_test_split(X_train_res, y_train_res, test_size=0.25, random_state=42)


In [14]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)


In [17]:
from catboost import CatBoostClassifier, Pool

MODEL_PATH = "catboost_model.cbm"


model = CatBoostClassifier()

model.load_model(MODEL_PATH)

In [19]:
y_pred =model.predict(X_test)
test_score = model.score(y_pred, y_test)
print("Результат на тестовой выборке:", test_score)

Результат на тестовой выборке: 0.4989794456778629


In [21]:
params_model = model.get_params()


In [23]:
from sklearn.metrics import accuracy_score
model_metric = accuracy_score(y_test, y_pred)
metric_type = "accuracy"


In [20]:
import mlflow
from mlflow.models import infer_signature
import mlflow.sklearn
mlflow.set_tracking_uri(uri='http://localhost:5001')

In [24]:
# Create a new MLflow Experiment
mlflow.set_experiment("MTS MLops_hw_1")

# Start an MLflow run
with mlflow.start_run():
    # Log model parameters
    mlflow.log_params(params_model)

    # Log model metric values
    mlflow.log_metric(metric_type, model_metric)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "First Catboost model, Light preprocessing")

    # Infer the model's signature
    signature = infer_signature(
        X_train,
        {
            "predict": model.predict(X_train),
            "predict_proba": model.predict_proba(X_train),
        },
    )

    # Log the model
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=signature,
        input_example=X_train,
        registered_model_name="catboost-first_attempt"
    )


2025/06/12 21:19:24 INFO mlflow.tracking.fluent: Experiment with name 'MTS MLops_hw_1' does not exist. Creating a new experiment.
/Users/katiegalaeva/miniconda3/envs/catboost_env/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/06/12 21:19:35 WARNING mlflow.models.model: `artifact_path` is deprecated. 

🏃 View run powerful-wasp-575 at: http://localhost:5001/#/experiments/1/runs/953026c9257a4227bc1429c9792db8a7
🧪 View experiment at: http://localhost:5001/#/experiments/1
